[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/02-PDF_reporte.ipynb)


📁 [Recursos del módulo (Google Drive)](https://drive.google.com/drive/folders/1t4VQB_6FL5wpJTPIw6dprTnCPt0uDjYg?usp=drive_link)


# 02 — Parsing: texto desde un PDF

Aquí el documento ya es **PDF con texto digital** (no fotos escaneadas): podemos extraer caracteres con **`pypdf`** (el proyecto actual que continúa a **PyPDF2**).

A esto se le llama **parsing**: convertir un documento con un formato específico (aquí, PDF) en texto plano que se pueda usar — sin buscar nada en particular todavía. Parsing responde "¿qué dice este documento?", no "¿cuál es su folio o su total?" (eso es *extraction*, y lo vemos en `03-OCRfacturas`).

## Qué documento usamos

**EY — *100 casos rentables de IA* (2026)** (PDF en español): informe de negocio sobre casos de uso de inteligencia artificial, con vocabulario **sectorial** (empresa, datos, modelos, valor, riesgos…) además de las palabras muy frecuentes del español. Es **sustancioso** para practicar lo mismo que en **`01_Texto_y_maquina`**: tokens, limpieza con regex, vocabulario y frecuencias.

---


## Carpeta `content/`

Este notebook (y el `03`) esperan una carpeta llamada **`content`**, junto al notebook, con los archivos de datos que se usan — igual que tendrías que armarla en Colab.

**En Colab:** panel de archivos (ícono de carpeta, izquierda) → clic derecho → *Nueva carpeta* → nómbrala `content` → arrastra ahí `ey_100_casos_rentables_ia_2026.pdf`, `factura_1.pdf` y `factura_digital.pdf` (las usamos al final).

**En local:** si clonaste el repo, ya existe `modulo_4/content/` con esos archivos.


## Instalación


In [ ]:
%pip install -q pypdf wordcloud


## Cargar el PDF


In [ ]:
from pathlib import Path

PDF_LOCAL = Path("content/ey_100_casos_rentables_ia_2026.pdf")
print(PDF_LOCAL.resolve())


## Parsing: extraer todo el texto con `pypdf`


In [ ]:
from pypdf import PdfReader

lector = PdfReader(str(PDF_LOCAL))
num_paginas = len(lector.pages)
print(f"Páginas: {num_paginas}")

fragmentos: list[str] = []
for i, pagina in enumerate(lector.pages):
    t = pagina.extract_text()
    if t:
        fragmentos.append(t)

texto_completo = "\n".join(fragmentos)
print(f"Caracteres extraídos (aprox.): {len(texto_completo):,}")
print("\n--- Muestra (primeros 1200 caracteres) ---\n")
print(texto_completo[:1200])


## Split: de texto completo a pedazos

El parsing te da **un solo bloque de texto**. Para un reporte de 24 páginas, muchas veces conviene trabajar por partes (por página, por sección) en vez de con todo junto — sobre todo si más adelante vas a buscar información específica ahí adentro o alimentarlo a un modelo con límite de contexto (lo vemos con detalle cuando lleguemos a RAG, en `10_RAG_fundamentos`).

`pypdf` ya nos da el texto por página por separado (es como lo armamos arriba, antes de juntarlo todo en `texto_completo`), así que "partir por página" es casi gratis:


In [ ]:
paginas_texto = [pagina.extract_text() or "" for pagina in lector.pages]

print(f"El reporte quedó partido en {len(paginas_texto)} páginas.")
print("\n--- Primeras líneas de la página 5 ---\n")
print(paginas_texto[4][:300])


## Limpieza y tokens (misma lógica que en `01_Texto_y_maquina`)

Pasamos a minúsculas, quitamos signos de puntuación frecuentes y partimos por espacios. En informes corporativos el PDF a veces inserta saltos raros; para un primer análisis basta esto.


In [ ]:
import re

texto_limpio = texto_completo.lower()
texto_limpio = re.sub(r"[.,;:!?¿¡'\"()\[\]{}—–-]", " ", texto_limpio)
texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()

palabras = texto_limpio.split()
print(f"Total de tokens (aprox.): {len(palabras):,}")
print(f"Palabras únicas (vocabulario aprox.): {len(set(palabras)):,}")


## Qué palabras se repiten más

Las más frecuentes suelen ser **funcionales** (artículos, preposiciones). Más abajo filtramos una lista corta de **stopwords** en español solo para ilustrar otras palabras más “de contenido”.


In [ ]:
from collections import Counter

frecuencias = Counter(palabras)
print("Las 25 más frecuentes:\n")
for palabra, n in frecuencias.most_common(25):
    print(f"  {n:>6}  {palabra}")


In [ ]:
# Stopwords mínimas (solo didácticas; una librería como NLTK trae listas más completas)
STOP = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "y", "o", "u", "de", "del", "al",
    "en", "que", "con", "por", "para", "como", "se", "su", "sus", "lo", "le", "les", "a",
    "no", "si", "ya", "más", "menos", "tan", "entre", "sobre", "ser", "es", "son", "fue",
    "ha", "han", "he", "hay", "está", "están", "este", "esta", "esto", "ese", "esa", "eso",
}

palabras_filtradas = [p for p in palabras if p not in STOP and len(p) > 2]
frec_filtrada = Counter(palabras_filtradas)
print("25 más frecuentes tras quitar algunas stopwords:\n")
for palabra, n in frec_filtrada.most_common(25):
    print(f"  {n:>5}  {palabra}")


## Nube de palabras

Visualizamos las frecuencias ya filtradas: cuanto más grande la palabra, más veces aparece en el texto del PDF.


In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud


def plot_wordcloud(contador, max_palabras=80):
    """Dibuja una nube a partir de un Counter (palabra → cuántas veces sale)."""
    frecuencias = dict(contador.most_common(500))
    nube = WordCloud(
        width=900,
        height=500,
        background_color="white",
        max_words=max_palabras,
        colormap="viridis",
    ).generate_from_frequencies(frecuencias)

    plt.figure(figsize=(11, 6))
    plt.imshow(nube, interpolation="bilinear")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


plot_wordcloud(frec_filtrada)


## Cierre

- **Parsing** (`pypdf`) convierte el PDF en texto plano — sin buscar nada en particular todavía.
- **Split** parte ese texto en pedazos (aquí, por página) para trabajar con partes en vez de con todo el bloque junto.
- El análisis de frecuencias conecta directamente con **vocabulario, repetición y límites del tokenizado "por espacios"** del notebook `01_Texto_y_maquina`.

**Otras ideas** (misma técnica, subiendo otro PDF con texto digital a `content/`): informes de **CEPAL**, **INEGI**, **Banco de México**, **ONU/PNUD**, u otros documentos de negocio o sector público — siempre revisa **licencia** y uso educativo.


## Parsing tiene un límite: necesita texto seleccionable

Todo lo anterior funcionó porque el PDF del reporte **ya traía texto seleccionable** (lo puedes marcar y copiar si lo abres normal). Pero no todos los PDF son así: uno escaneado, o una foto guardada como PDF, son solo una **imagen** — no hay texto que copiar, aunque a simple vista se vea igual.

Probemos con `factura_1.pdf`: es un PDF, pero por dentro es una imagen (así se generó).


In [ ]:
lector_factura = PdfReader("content/factura_1.pdf")
texto_factura = lector_factura.pages[0].extract_text()
print(f"Caracteres extraídos: {len(texto_factura)}")
print(repr(texto_factura[:200]))


**Casi nada.** La factura tiene toda la información — folio, fecha, total — pero como píxeles, no como caracteres. `pypdf` solo sabe leer la capa de texto de un PDF; si no existe, no hay nada que parsear.

Para estos casos hace falta **OCR** (reconocimiento óptico de caracteres): desde motores clásicos (Tesseract, EasyOCR, PaddleOCR) hasta modelos modernos especializados en documentos, como **Falcon-OCR** de Hugging Face, o servicios en la nube como **LlamaExtract**. Eso es justo lo que comparamos en **`03-OCRfacturas`**.


## Y el contraste: un PDF sencillo que sí se puede parsear

Para que quede claro que el problema es el documento, no `pypdf`: probemos con `factura_digital.pdf` — una factura de ejemplo con texto real adentro (no una imagen).


In [ ]:
lector_digital = PdfReader("content/factura_digital.pdf")
texto_digital = lector_digital.pages[0].extract_text()
print(f"Caracteres extraídos: {len(texto_digital)}")
print(texto_digital)


**Ahí sí funcionó, completo y al instante.** Mismo código de `pypdf`, dos resultados distintos — la diferencia siempre está en si el PDF trae texto seleccionable o no. Antes de pensar en OCR, vale la pena probar esto primero: es gratis y toma un segundo.


## Bonus opcional — un ejemplo real, no inventado por nosotros

Todo lo anterior lo armamos nosotros (la factura, el reporte). Aquí un caso real: una página escaneada de ***La sociedad de la mente*** (Marvin Minsky) — texto perfectamente legible a simple vista, pero es solo una imagen.

Es un libro con derechos de autor, así que **este archivo no viene en el repo** (no se sube a GitHub). Si tienes tu propia copia en `content/`, ajusta el nombre abajo y pruébala; si no, sáltate esta celda.


In [ ]:
from PIL import Image

RUTA_ESCANEO = Path("content/minsky_pagina.png")  # cambia el nombre si el tuyo es distinto

if RUTA_ESCANEO.exists():
    ruta_pdf_escaneo = Path("pagina_escaneada.pdf")
    Image.open(RUTA_ESCANEO).convert("RGB").save(ruta_pdf_escaneo, "PDF")
    texto_libro = PdfReader(str(ruta_pdf_escaneo)).pages[0].extract_text()
    print(f"Caracteres extraídos: {len(texto_libro)}")
    print(repr(texto_libro[:200]))
else:
    print("No tienes ese archivo en content/ — sáltate esta celda.")


Mismo resultado que con la factura: se ve como texto, pero `pypdf` no encuentra nada que parsear — porque no lo hay, es una foto. Ni el libro más claro y mejor escrito se salva de este límite si solo tienes la imagen.
